In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        

In [2]:
import pandas as pd

df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/competitions/playground-series-s6e4/train.csv'

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
num_cols = ['Soil_Moisture','Temperature_C','Rainfall_mm', 'Wind_Speed_kmh','Soil_pH','Humidity']

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers)} outliers")

In [ ]:
missing_values = df.isnull().sum().to_frame(name='Missing Count')
missing_values['Percentage (%)'] = (missing_values['Missing Count'] / len(df)) * 100
print("--- Missing Value Report ---")
print(missing_values)

In [ ]:
cat_cols = ['Soil_Type','Crop_Type','Crop_Growth_Stage','Season',
            'Irrigation_Type','Water_Source','Mulching_Used','Region']

for col in cat_cols:
    print(f"--- Analysis for: {col} ---")
    print(df.groupby(col)['Irrigation_Need'].value_counts(normalize=True).round(2))
    print("\n")

In [ ]:
from scipy.stats import chi2_contingency
import numpy as np
import matplotlib.pyplot as plt

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    return np.sqrt(phi2 / min(k-1, r-1))

results = {}
for col in cat_cols:
    results[col] = cramers_v(df[col], df['Irrigation_Need'])

cramers_df = pd.Series(results).sort_values(ascending=False)
print(cramers_df)

plt.figure(figsize=(10, 6))
cramers_df.plot(kind='bar', color='#3b82f6')
plt.title("Cramér's V — Categorical Features vs Irrigation Need")
plt.ylabel("Cramér's V Score")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
num_cols_plot = ['Soil_Moisture', 'Temperature_C', 'Rainfall_mm', 
                 'Wind_Speed_kmh', 'Soil_pH', 'Humidity']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols_plot):
    df.boxplot(column=col, by='Irrigation_Need', ax=axes[i])
    axes[i].set_title(f'{col} Distribution')
    axes[i].set_xlabel('')

plt.suptitle('Feature Distribution by Irrigation Need Target')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
feature_counts = {'Soil_Moisture': 511, 'Temperature_C': 940, 'Wind_Speed_kmh': 596, 'Unique Total': 1968}
high_total = len(df[df['Irrigation_Need'] == 'High'])

labels = list(feature_counts.keys())
counts = list(feature_counts.values())
pcts = [c / high_total * 100 for c in counts]
colors = ['#378ADD', '#378ADD', '#378ADD', '#D85A30']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, counts, color=colors, edgecolor='white', linewidth=0.5)

for bar, count, pct in zip(bars, counts, pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f'{count}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=11)

ax.set_title('Boxplot Black Dot Outliers — High Class', fontsize=14)
ax.set_ylabel('Row Count')
ax.set_ylim(0, max(counts) * 1.2)
ax.axvline(x=2.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.text(2.6, max(counts) * 1.1, 'combined', fontsize=9, color='gray')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
high_df = df[df['Irrigation_Need'] == 'High']
Q3 = high_df['Soil_Moisture'].quantile(0.75)
IQR = high_df['Soil_Moisture'].quantile(0.75) - high_df['Soil_Moisture'].quantile(0.25)
black_dots = high_df[high_df['Soil_Moisture'] > Q3 + 1.5 * IQR]
print(black_dots['Crop_Growth_Stage'].value_counts(normalize=True))
print(black_dots['Mulching_Used'].value_counts(normalize=True))

In [ ]:
df[num_cols_plot].hist(bins=30, figsize=(15, 10), color='#3b82f6', edgecolor='black')
plt.suptitle("Distribution of Numerical Features", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import seaborn as sns

numeric_df = df.select_dtypes(include='number')

plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Numerical Feature Correlation Heatmap")
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_corr = numeric_df.copy()
df_corr['Irrigation_Need_Encoded'] = LabelEncoder().fit_transform(df['Irrigation_Need'])

plt.figure(figsize=(12, 8))
sns.heatmap(df_corr.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation with Encoded Target")
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
counts = df['Irrigation_Need'].value_counts()
percentages = df['Irrigation_Need'].value_counts(normalize=True) * 100
colors = ['#3b82f6', '#eab308', '#ef4444']

plt.figure(figsize=(8, 6))
ax = sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette=colors, legend=False)

for i, p in enumerate(ax.patches):
    height = p.get_height()
    text = f'{int(height)}\n({percentages.values[i]:.1f}%)'
    ax.annotate(text, 
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='center', 
                xytext=(0, 15), 
                textcoords='offset points',
                fontsize=11, fontweight='bold')

plt.title("Irrigation Need - Class Distribution & Imbalance Check", fontsize=14, pad=20)
plt.xlabel("Irrigation Need Class")
plt.ylabel("Number of Samples")
plt.ylim(0, counts.max() * 1.2)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()

In [ ]:
sns.pairplot(df.sample(2000), vars=num_cols_plot, hue='Irrigation_Need', palette={'Low': '#3b82f6', 'Medium': '#eab308', 'High': '#ef4444'}, diag_kind='kde')
plt.suptitle("Bivariate Analysis - Numerical Features (Sample n=2000)", y=1.02)
plt.show()

In [ ]:
high_df = df[df['Irrigation_Need'] == 'High']
normal_high = df[(df['Irrigation_Need'] == 'High') & 
                 (df['Soil_Moisture'] < 25)]
bio_high = df[(df['Irrigation_Need'] == 'High') & 
              (df['Temperature_C'] < 25) & 
              (df['Soil_Moisture'] > 35)]

print(f"Total High: {len(high_df)}")
print(f"Normal High (dry soil): {len(normal_high)} ({len(normal_high)/len(high_df)*100:.1f}%)")
print(f"Biological anomaly: {len(bio_high)} ({len(bio_high)/len(high_df)*100:.1f}%)")

print("\n--- Growth Stage: HIGH vs ALL DATA ---")
stage_high = high_df['Crop_Growth_Stage'].value_counts(normalize=True).rename('High_%')
stage_all = df['Crop_Growth_Stage'].value_counts(normalize=True).rename('All_%')
stage_compare = pd.concat([stage_high, stage_all], axis=1).round(3)
stage_compare['Lift'] = (stage_compare['High_%'] / stage_compare['All_%']).round(2)
print(stage_compare)

print("\n--- High Rate per Growth Stage (%) ---")
stage_high_rate = df.groupby('Crop_Growth_Stage')['Irrigation_Need'].apply(
    lambda x: (x == 'High').mean() * 100
).round(2).sort_values(ascending=False)
print(stage_high_rate)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(x='Temperature_C', y='Soil_Moisture', 
                hue='Irrigation_Need', data=df.sample(5000, random_state=42),
                palette={'Low': '#3b82f6', 'Medium': '#eab308', 'High': '#ef4444'},
                alpha=0.5, ax=axes[0])
axes[0].set_title("Irrigation Need by Temp vs Moisture")

sns.scatterplot(x='Temperature_C', y='Soil_Moisture',
                hue='Crop_Growth_Stage', data=high_df.sample(min(2000, len(high_df)), random_state=42),
                alpha=0.6, ax=axes[1])
axes[1].set_title("HIGH Class — Growth Stage Distribution")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import lightgbm as lgb
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')
orig  = pd.read_csv('/kaggle/input/datasets/cdeotte/s6e4-the-original-dataset/irrigation_prediction.csv')
sub   = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv')

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"Original shape: {orig.shape}")
print(f"Target distribution:")
print(train['Irrigation_Need'].value_counts(normalize=True).round(4))


In [ ]:
if 'Irrigation_Need' in orig.columns:
    orig = orig.drop(columns=['id'], errors='ignore')
    train_combined = pd.concat([train, orig], ignore_index=True)
    print(f"Orijinal veri eklendi → combined shape: {train_combined.shape}")
else:
    train_combined = train.copy()
    print("Orijinal veri uyumsuz, sadece train kullanılıyor.")


In [ ]:
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
inv_target = {v: k for k, v in target_map.items()}

y      = train_combined['Irrigation_Need'].map(target_map)
X_all  = train_combined.drop(columns=['id', 'Irrigation_Need'])
X_test = test.drop(columns=['id'])


In [ ]:
def feature_engineering(df):
    df = df.copy()

    # Aktif büyüme aşaması bayrağı (Flowering/Vegetative = High riski ~6.4%)
    df['Is_Active_Growth'] = df['Crop_Growth_Stage'].isin(
        ['Flowering', 'Vegetative']).astype(int)

    # Mulching × Aktif büyüme etkileşimi
    df['Mulching_x_ActiveGrowth'] = df['Is_Active_Growth'] * \
        (df['Mulching_Used'] == 'No').astype(int)

    # Buharlaşma Baskısı Proxy
    df['Evap_Stress'] = (df['Temperature_C'] - df['Humidity']
                         - df['Rainfall_mm'] / 10)

    # Su Açığı bayrağı
    prev_med  = df['Previous_Irrigation_mm'].median()
    moist_med = df['Soil_Moisture'].median()
    df['Water_Deficit'] = np.where(
        (df['Previous_Irrigation_mm'] < prev_med) &
        (df['Soil_Moisture'] < moist_med), 1, 0)

    # Isı Yükü: sıcaklık × güneşlenme saati
    df['Heat_Load'] = df['Temperature_C'] * df['Sunlight_Hours']

    # Toprak verimliliği proxy
    df['Soil_Fertility'] = df['Soil_Moisture'] * df['Organic_Carbon']

    # Rüzgar Buharlaşma İndeksi
    df['Wind_Evap_Index'] = df['Wind_Speed_kmh'] * (100 - df['Humidity']) / 100

    # Yağış Kategorisi
    df['Rainfall_Cat'] = pd.cut(df['Rainfall_mm'],
                                 bins=[-np.inf, 20, 50, 100, np.inf],
                                 labels=[0, 1, 2, 3]).astype(float)
    return df

X_all  = feature_engineering(X_all)
X_test = feature_engineering(X_test)

print(f"Özellik sayısı (ham)               : {train.shape[1] - 2}")
print(f"Özellik sayısı (mühendislik sonrası): {X_all.shape[1]}")


In [ ]:
cat_cols = X_all.select_dtypes(include='object').columns.tolist()
print(f"Kategorik kolonlar: {cat_cols}")

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_all[cat_cols]  = oe.fit_transform(X_all[cat_cols])
X_test[cat_cols] = oe.transform(X_test[cat_cols])

X_all  = X_all.astype(float)
X_test = X_test.astype(float)
print(f"Final feature matrix shape: {X_all.shape}")


In [ ]:
import numpy as np
import pandas as pd
import optuna
import catboost as cb
from catboost import CatBoostClassifier, Pool
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:

print("Scikit-learn Pipeline")

 

train_raw = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test_raw  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')
orig_raw  = pd.read_csv('/kaggle/input/datasets/cdeotte/s6e4-the-original-dataset/irrigation_prediction.csv')
 
if 'Irrigation_Need' in orig_raw.columns:
    orig_raw = orig_raw.drop(columns=['id'], errors='ignore')
    train_combined_raw = pd.concat([train_raw, orig_raw], ignore_index=True)
else:
    train_combined_raw = train_raw.copy()
 
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
y_raw = train_combined_raw['Irrigation_Need'].map(target_map)
X_raw = train_combined_raw.drop(columns=['id', 'Irrigation_Need'])
X_test_raw = test_raw.drop(columns=['id'])
 
 
def feature_engineering(df):
    """Kişi 2'nin feature engineering fonksiyonu (değişmedi)."""
    df = df.copy()
    df['Is_Active_Growth'] = df['Crop_Growth_Stage'].isin(
        ['Flowering', 'Vegetative']).astype(int)
    df['Mulching_x_ActiveGrowth'] = df['Is_Active_Growth'] * \
        (df['Mulching_Used'] == 'No').astype(int)
    df['Evap_Stress'] = (df['Temperature_C'] - df['Humidity']
                         - df['Rainfall_mm'] / 10)
    prev_med  = df['Previous_Irrigation_mm'].median()
    moist_med = df['Soil_Moisture'].median()
    df['Water_Deficit'] = np.where(
        (df['Previous_Irrigation_mm'] < prev_med) &
        (df['Soil_Moisture'] < moist_med), 1, 0)
    df['Heat_Load']       = df['Temperature_C'] * df['Sunlight_Hours']
    df['Soil_Fertility']  = df['Soil_Moisture'] * df['Organic_Carbon']
    df['Wind_Evap_Index'] = df['Wind_Speed_kmh'] * (100 - df['Humidity']) / 100
    df['Rainfall_Cat']    = pd.cut(df['Rainfall_mm'],
                                   bins=[-np.inf, 20, 50, 100, np.inf],
                                   labels=[0, 1, 2, 3]).astype(float)
    return df
    
X_fe = feature_engineering(X_raw)
X_test_fe = feature_engineering(X_test_raw)
cat_cols = X_fe.select_dtypes(include='object').columns.tolist()
num_cols = X_fe.select_dtypes(exclude='object').columns.tolist()
 
# Pipeline: sadece OrdinalEncoder adımı (FE saf bir transform, sklearn'a gerek yok)
preprocessing_pipeline = Pipeline([
    ('encoder', OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])
 
# Pipeline ile encode (CV içinde kullanım örneği)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
pipeline_scores = []
for fold, (trn_idx, val_idx) in enumerate(skf.split(X_fe, y_raw), 1):
    X_tr_raw, X_val_raw = X_fe.iloc[trn_idx].copy(), X_fe.iloc[val_idx].copy()
 
 
    X_tr_raw[cat_cols]  = preprocessing_pipeline.fit_transform(X_tr_raw[cat_cols])
    X_val_raw[cat_cols] = preprocessing_pipeline.transform(X_val_raw[cat_cols])
 
    X_tr_raw  = X_tr_raw.astype(float)
    X_val_raw = X_val_raw.astype(float)
 
  
    quick_lgb = lgb.LGBMClassifier(
        objective='multiclass', num_class=3,
        n_estimators=300, learning_rate=0.1,
        num_leaves=63, random_state=42,
        n_jobs=-1, verbose=-1
    )
    quick_lgb.fit(X_tr_raw, y_raw.iloc[trn_idx])
    preds = quick_lgb.predict(X_val_raw)
    pipeline_scores.append(accuracy_score(y_raw.iloc[val_idx], preds))
    print(f"  Pipeline Fold {fold}  Accuracy: {pipeline_scores[-1]:.5f}")
 
print(f"Pipeline CV — Mean: {np.mean(pipeline_scores):.5f}  "
      f"Std: {np.std(pipeline_scores):.5f}")
print("✓ Pipeline kuruldu — data leakage riski ortadan kalktı.\n")
 
# Tam encode (tüm veri üzerinde, final modeller için)
oe_final = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_all = X_fe.copy()
X_test_all = X_test_fe.copy()
X_all[cat_cols]      = oe_final.fit_transform(X_all[cat_cols])
X_test_all[cat_cols] = oe_final.transform(X_test_all[cat_cols])
X_all      = X_all.astype(float)
X_test_all = X_test_all.astype(float)
y = y_raw
 

In [ ]:
lgb_params = {
    'objective'        : 'multiclass',
    'num_class'        : 3,
    'metric'           : 'multi_logloss',
    'n_estimators'     : 1000,
    'learning_rate'    : 0.05,
    'num_leaves'       : 127,
    'min_child_samples': 50,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 0.1,
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbose'          : -1,
}

xgb_params = {
    'objective'           : 'multi:softprob',
    'num_class'           : 3,
    'eval_metric'         : 'mlogloss',
    'n_estimators'        : 800,
    'learning_rate'       : 0.05,
    'max_depth'           : 7,
    'min_child_weight'    : 5,
    'subsample'           : 0.8,
    'colsample_bytree'    : 0.8,
    'gamma'               : 0.1,
    'reg_alpha'           : 0.1,
    'reg_lambda'          : 1.0,
    'random_state'        : 42,
    'n_jobs'              : -1,
    'verbosity'           : 0,
    'early_stopping_rounds': 50,
}


In [ ]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_lgb  = np.zeros((len(X_all), 3))
oof_xgb  = np.zeros((len(X_all), 3))
test_lgb = np.zeros((len(X_test), 3))
test_xgb = np.zeros((len(X_test), 3))
fold_scores_lgb = []
fold_scores_xgb = []

# ── LightGBM ──
for fold, (trn_idx, val_idx) in enumerate(skf.split(X_all, y), 1):
    X_tr, X_val = X_all.iloc[trn_idx], X_all.iloc[val_idx]
    y_tr, y_val = y.iloc[trn_idx],     y.iloc[val_idx]

    model_lgb = lgb.LGBMClassifier(**lgb_params)
    model_lgb.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                              lgb.log_evaluation(0)])

    oof_lgb[val_idx]  = model_lgb.predict_proba(X_val)
    test_lgb         += model_lgb.predict_proba(X_test) / N_FOLDS
    score = accuracy_score(y_val, oof_lgb[val_idx].argmax(axis=1))
    fold_scores_lgb.append(score)
    print(f"  LGB Fold {fold}  Accuracy: {score:.5f}")

print(f"LightGBM CV — Mean: {np.mean(fold_scores_lgb):.5f}  Std: {np.std(fold_scores_lgb):.5f}")



In [ ]:
# ── XGBoost ──
for fold, (trn_idx, val_idx) in enumerate(skf.split(X_all, y), 1):
    X_tr, X_val = X_all.iloc[trn_idx], X_all.iloc[val_idx]
    y_tr, y_val = y.iloc[trn_idx],     y.iloc[val_idx]

    model_xgb = xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=3,
        eval_metric='mlogloss',
        n_estimators=800,
        learning_rate=0.05,
        max_depth=7,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=0,
        early_stopping_rounds=50
    )
    model_xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    oof_xgb[val_idx]  = model_xgb.predict_proba(X_val)
    test_xgb         += model_xgb.predict_proba(X_test) / N_FOLDS
    score = accuracy_score(y_val, oof_xgb[val_idx].argmax(axis=1))
    fold_scores_xgb.append(score)
    print(f"  XGB Fold {fold}  Accuracy: {score:.5f}")

print(f"XGBoost   CV — Mean: {np.mean(fold_scores_xgb):.5f}  Std: {np.std(fold_scores_xgb):.5f}")

In [ ]:

print("CatBoost - 5-Fold CV + OOF")



cat_col_indices = [X_fe.columns.get_loc(c) for c in cat_cols]

N_FOLDS = 5
oof_cb   = np.zeros((len(X_all), 3))
test_cb  = np.zeros((len(X_test_all), 3))
fold_scores_cb = []

cb_params = dict(
    loss_function='MultiClass',
    eval_metric='Accuracy',
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    bagging_temperature=0.5,
    random_seed=42,
    thread_count=-1,
    verbose=0,
    early_stopping_rounds=50,
    cat_features=cat_col_indices,
)

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_fe, y), 1):
    X_tr, X_val = X_fe.iloc[trn_idx], X_fe.iloc[val_idx]
    y_tr, y_val = y.iloc[trn_idx],    y.iloc[val_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_col_indices)
    val_pool   = Pool(X_val, y_val, cat_features=cat_col_indices)
    test_pool  = Pool(X_test_fe, cat_features=cat_col_indices)

    model_cb = CatBoostClassifier(**cb_params)
    model_cb.fit(train_pool, eval_set=val_pool)

    oof_cb[val_idx] = model_cb.predict_proba(val_pool)
    test_cb        += model_cb.predict_proba(test_pool) / N_FOLDS

    score = accuracy_score(y_val, oof_cb[val_idx].argmax(axis=1))
    fold_scores_cb.append(score)
    print(f"  CB  Fold {fold}  Accuracy: {score:.5f}")

print(f"CatBoost  CV — Mean: {np.mean(fold_scores_cb):.5f}  "
      f"Std: {np.std(fold_scores_cb):.5f}\n")

In [ ]:
# C) Optuna — 50 trial sonucu (hardcode)
best_lgb_params = {
    'objective'        : 'multiclass',
    'num_class'        : 3,
    'metric'           : 'multi_logloss',
    'n_estimators'     : 1500,
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbose'          : -1,
    'learning_rate'    : 0.031807813819639215,
    'num_leaves'       : 180,
    'min_child_samples': 68,
    'subsample'        : 0.7867764629226751,
    'colsample_bytree' : 0.8215293581171331,
    'reg_alpha'        : 0.3315205091823159,
    'reg_lambda'       : 0.6664948014256238,
    'max_depth'        : 6,
}

oof_lgb_opt  = np.zeros((len(X_all), 3))
test_lgb_opt = np.zeros((len(X_test_all), 3))
fold_scores_lgb_opt = []

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_all, y), 1):
    X_tr, X_val = X_all.iloc[trn_idx], X_all.iloc[val_idx]
    y_tr, y_val = y.iloc[trn_idx],     y.iloc[val_idx]
    m = lgb.LGBMClassifier(**best_lgb_params)
    m.fit(X_tr, y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(0)])
    oof_lgb_opt[val_idx] = m.predict_proba(X_val)
    test_lgb_opt        += m.predict_proba(X_test_all) / N_FOLDS
    score = accuracy_score(y_val, oof_lgb_opt[val_idx].argmax(axis=1))
    fold_scores_lgb_opt.append(score)
    print(f"  LGB-Opt Fold {fold}  Accuracy: {score:.5f}")

print(f"LGB (Optuna)  CV — Mean: {np.mean(fold_scores_lgb_opt):.5f}  "
      f"Std: {np.std(fold_scores_lgb_opt):.5f}")

In [ ]:
best_w, best_oof_score = 0.5, 0.0
for w in np.arange(0.3, 0.8, 0.05):
    blend = w * oof_lgb + (1 - w) * oof_xgb
    sc = accuracy_score(y, blend.argmax(axis=1))
    if sc > best_oof_score:
        best_oof_score = sc
        best_w = w

print(f"En iyi ağırlık → LGB: {best_w:.2f}  XGB: {1-best_w:.2f}")
print(f"Ensemble OOF Accuracy: {best_oof_score:.5f}")

oof_ensemble  = best_w * oof_lgb  + (1 - best_w) * oof_xgb
test_ensemble = best_w * test_lgb + (1 - best_w) * test_xgb
oof_preds     = oof_ensemble.argmax(axis=1)
test_preds    = test_ensemble.argmax(axis=1)


In [ ]:
print(classification_report(y, oof_preds, target_names=['Low', 'Medium', 'High']))

In [ ]:
cm = confusion_matrix(y, oof_preds)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'], ax=ax)
ax.set_title('OOF Confusion Matrix — Ensemble', fontsize=13)
ax.set_ylabel('Gerçek Sınıf')
ax.set_xlabel('Tahmin Edilen Sınıf')
plt.tight_layout()
plt.show()


In [ ]:
feat_imp = pd.Series(model_lgb.feature_importances_, index=X_all.columns)
feat_imp = feat_imp.sort_values(ascending=True).tail(20)

plt.figure(figsize=(9, 7))
feat_imp.plot(kind='barh', color='#3b82f6')
plt.title('Top 20 Feature Importance — LightGBM (son fold)', fontsize=13)
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


In [ ]:
results = pd.DataFrame({
    'Model'   : ['LightGBM', 'XGBoost', 'Ensemble'],
    'CV Mean' : [np.mean(fold_scores_lgb), np.mean(fold_scores_xgb), best_oof_score],
    'CV Std'  : [np.std(fold_scores_lgb),  np.std(fold_scores_xgb),  0.0]
})
print(results.to_string(index=False))


In [ ]:
best_w = 0.5
test_ensemble = best_w * test_lgb + (1 - best_w) * test_xgb
test_preds    = test_ensemble.argmax(axis=1)

inv_target = {0: 'Low', 1: 'Medium', 2: 'High'}
sub = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv')
sub['Irrigation_Need'] = pd.Series(test_preds).map(inv_target)
sub.to_csv('submission.csv', index=False)
print("--- Submission Dağılımı ---")
print(sub['Irrigation_Need'].value_counts())
print("submission.csv kaydedildi.")

In [ ]:

print("ÖZET — Model Karşılaştırması")

 
try:
    from sklearn.metrics import accuracy_score as acc
 
    
    lgb_baseline   = accuracy_score(y, oof_lgb.argmax(axis=1))
    xgb_baseline   = accuracy_score(y, oof_xgb.argmax(axis=1))
    blend_baseline = accuracy_score(y, (0.5*oof_lgb + 0.5*oof_xgb).argmax(axis=1))
    summary = pd.DataFrame({
        'Model': [
            'LightGBM (orijinal)',
            'XGBoost  (orijinal)',
            'Blend    (orijinal)',
            'CatBoost (yeni)',
            'LGB-Optuna (yeni)',
            
        ],
        'OOF Accuracy': [
            lgb_baseline,
            xgb_baseline,
            blend_baseline,
            np.mean(fold_scores_cb),
            np.mean(fold_scores_lgb_opt),
           
        ]
    })
    summary['Delta vs Blend'] = summary['OOF Accuracy'] - blend_baseline
    summary = summary.sort_values('OOF Accuracy', ascending=False)
    print(summary.to_string(index=False))
 
except Exception as e:
    print(f"Özet tablo oluşturulamadı: {e}")
 
print("\nTamamlandı ✓")